# LLM runner: staged tests (model: gpt-6-astra)

Run top to bottom. Stages 1-3 cost nothing. Stages 4-6 call the API and are **off by default**: change the `RUN_...` switch to `True` in that cell to run it.

| Stage | What it tests | Cost |
|---|---|---|
| 0 | Setup and config | free |
| 1 | Inputs are identical across conditions (dry run) | free |
| 2 | Whole pipeline with a fake model, including the citation check | free |
| 3 | Key works and `gpt-6-astra` is visible to it | free |
| 4 | One tiny real call (plain text) | a few tokens |
| 5 | Which settings the model accepts (reasoning effort, JSON schema) | a few small calls |
| 6 | Pilot on A0 and A3, then read the output and check citations | real cost |

Pilot runs are rehearsals. They are saved in `runs_pilot/` and must not be counted in the study.

## 0. Setup

In [ ]:
import sys, json, time
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "runner"))
import importlib, llm_runner as L
importlib.reload(L)
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
cfg = L.load_config(ROOT / "config.json")
MODEL = cfg["model"]
print("model:", MODEL, "| effort:", cfg["reasoning_effort"], "| max_output_tokens:", cfg["max_output_tokens"],
      "| json schema:", cfg["use_json_schema"], "| frozen:", cfg["frozen"])
assert not MODEL.startswith("REPLACE"), "set 'model' in config.json"

## 1. Dry run: are the five conditions built consistently?
Context and system prompt must be identical in all conditions, and each evidence package must appear unchanged inside A4. Evidence packages marked PLACEHOLDER (A1, A2) are expected to show up here. Pilots may use them, formal runs may not.

In [ ]:
checks = L.consistency_checks(cfg)
for name, ok, note in checks:
    print("PASS" if ok else "FAIL", name, f"[{note}]")
rows = []
for c in cfg["conditions"]:
    p = L.build_prompt(cfg, c)
    rows.append(dict(condition=c, packages=cfg["conditions"][c], approx_tokens=p["input_chars"] // 4,
                     ids_available=len(p["available_ids"]), placeholders=p["placeholders"], hash=p["input_sha256"][:12]))
display(pd.DataFrame(rows))
assert all(ok for _, ok, _ in checks), "fix the FAIL lines above before going further"

**What this should show:** every line PASS, and a different hash per condition (the evidence differs), but the same context and system prompt. Anything FAIL means the inputs are not comparable across conditions.

## 2. Fake-model rehearsal (free)
The fake model deliberately cites one real evidence ID and one invented one (`STRUCT-99`). A working pipeline should save the run and flag `STRUCT-99` as unknown.

In [ ]:
rows = L.run_batch(cfg, ["A0", "A3"], repeats=1, mode="pilot", mock=True)
display(pd.DataFrame(rows)[["run_id", "condition", "status", "n_hypotheses", "unknown_ids"]])
rep = pd.DataFrame(L.citation_report("mock"))
display(rep[["condition", "cited_ids", "unknown_ids"]].tail(2))
assert rep.tail(2).unknown_ids.str.contains("STRUCT-99").all(), "citation check did not catch the invented ID"
print("OK: pipeline saved the runs and the citation check caught STRUCT-99")

## 3. Does the key work, and can it see `gpt-6-astra`? (free)
Only asks the API for the model record. Nothing is generated and the key is never printed.

In [ ]:
client = L.get_client()
m = client.models.retrieve(MODEL)
print("visible:", m.id, "| owned_by:", m.owned_by)

## 4. One tiny real call (plain text, no schema)
Checks that a request to the model works at all. Uses a few hundred tokens.

In [ ]:
RUN_SMOKE = False     # set True to spend a few tokens
if RUN_SMOKE:
    t0 = time.time()
    r = client.responses.create(model=MODEL, input="Reply with exactly: OK", max_output_tokens=2000)
    print("status:", r.status, "| model returned:", r.model, "| seconds:", round(time.time() - t0, 1))
    print("text:", r.output_text)
    print("usage: in", r.usage.input_tokens, "out", r.usage.output_tokens)

## 5. Which settings does the model accept?
The runner sends a reasoning-effort setting and a JSON schema. This stage tries each one on a tiny prompt and records whether the model accepts it. If one fails, change that setting in `config.json` (`reasoning_effort`, `use_json_schema`) before the pilot.

In [ ]:
RUN_SETTINGS = False  # set True to run (several very small calls)
if RUN_SETTINGS:
    schema = {"type": "object", "properties": {"answer": {"type": "string"}}, "required": ["answer"], "additionalProperties": False}
    tests = [("no extras", {})]
    for e in ["low", "medium", "high"]:
        tests.append((f"effort={e}", {"reasoning": {"effort": e}}))
    tests.append(("json schema", {"text": {"format": {"type": "json_schema", "name": "t", "schema": schema, "strict": True}}}))
    out = []
    for name, extra in tests:
        try:
            r = client.responses.create(model=MODEL, input="Give a one-word answer: what colour is the sky?",
                                        max_output_tokens=3000, **extra)
            out.append(dict(test=name, accepted=True, status=r.status, out_tokens=r.usage.output_tokens, detail=r.output_text[:40]))
        except Exception as e:
            out.append(dict(test=name, accepted=False, status="error", out_tokens=None, detail=f"{type(e).__name__}: {str(e)[:150]}"))
    display(pd.DataFrame(out))

**How to read it:** `accepted = True` for every row means the runner's defaults are fine. A row with `accepted = False` names the setting to change. If every effort level is rejected, set `"reasoning_effort": null` in `config.json`.

## 6. Pilot with the real model (A0 and A3, one repeat each)
A0 has no evidence (baseline). A3 has the structure evidence (STRUCT-01 to 08). Results go to `runs_pilot/`. This is the first test of the full runner against the real API.

In [ ]:
RUN_PILOT = False     # set True to spend API credits
if RUN_PILOT:
    rows = L.run_batch(cfg, ["A0", "A3"], repeats=1, mode="pilot")
    display(pd.DataFrame(rows)[["run_id", "condition", "model_returned", "status", "input_tokens", "output_tokens", "reasoning_tokens", "unknown_ids", "error"]])

**What to check:** `status` is `completed` (not `format_problem`, `parse_error` or `error`), `model_returned` is the model you asked for, and `unknown_ids` is empty. If `status` is `incomplete`, raise `max_output_tokens` in `config.json`: reasoning tokens count toward it.

## 6b. Read what the model wrote

In [ ]:
for r in L.load_runs("pilot"):
    print("=" * 100, "\n", r["meta"]["run_id"], "|", r["meta"]["model_returned"], "|", r["meta"]["status"])
    for i, h in enumerate(r["output"]["hypotheses"], 1):
        print(f"\n  H{i}: {h['title']}  (confidence: {h['confidence']})")
        print("   Explanation:", h["question_and_explanation"])
        print("   Cites:", h["supporting_evidence_ids"], "| conflicts:", h["conflicting_evidence_ids"])
        print("   Test:", h["proposed_test"])
        print("   Would weaken it:", h["falsifying_outcome"])

## 6c. Citation check on the real pilot

In [ ]:
rep = pd.DataFrame(L.citation_report("pilot"))
display(rep[["run_id", "condition", "hypothesis", "n_cited", "unknown_ids", "title"]] if len(rep) else "no pilot outputs yet")

**What to look for:** A0 should cite only background (`CTX-`) records, since it has no evidence. A3 should cite `STRUCT-` records and stay within what they say. Any ID in `unknown_ids` means the model cited something that was not in its input. Also read the A3 text against the structure evidence for overclaiming (for example, treating a confidence score as proof of function).